# Lesson 12 Lab — Automatic Prefix Caching

**Puzzle:** When can a shared system prompt skip Prefill work, and when is the cache key different?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

Chat, document analysis, and few-shot workloads often repeat a long prefix. Automatic prefix caching can reuse KV blocks for exactly matching token prefixes, but it cannot reuse new suffix computation and it is not a semantic similarity cache.


## 0. Predict before running

1. Predict which request reports cached tokens.
2. Explain why changing one early token destroys downstream prefix matches.
3. Choose a workload where APC should not be enabled solely for speed.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

A native engine with prefix caching enabled serves a cold shared prefix, a warm exact prefix, and a one-token-mutated prefix. The lab retains cached-token fields exposed by RequestOutput and elapsed time for each case.

- Reuse requires token-exact prefix identity.
- Only cached Prefill blocks are skipped; Decode is unchanged.
- A hit-rate metric needs a request distribution and eviction window.


## 2. Derive the mechanism

A cache key covers token content and additional factors that affect KV validity. Matching full blocks can be referenced by another request; the final partial block and new suffix still require work. Hash lookup changes scheduling cost but not output semantics. Eviction and cache capacity determine whether a theoretical hit remains resident.

### Mechanism at a glance

```mermaid
flowchart LR
  P["tokenized prefix"] --> H["block hash lookup"]
  H --> M{"valid block match?"}
  M -->|"yes"| R["reference cached KV blocks"]
  M -->|"no"| C["compute Prefill blocks"]
  R --> S["compute new suffix"]
  C --> S
  S --> D["Decode normally"]
```

### Walk it step by step

1. **Tokenize deterministically.** Cache identity starts from exact token blocks.
2. **Look up full blocks.** Only valid resident matches can be referenced.
3. **Compute the remainder.** Unmatched suffix and partial blocks still run Prefill.
4. **Measure hit value.** Pair cached-token counters with latency and eviction behavior.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 12
LESSON_TITLE = 'Automatic Prefix Caching'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260824
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | cold shared prefix |
| Candidate | warm exact reuse and a near-match control |
| Held constant | engine instance, prefix length, suffix, sampling, maximum output, and GPU |
| Measurements | cached tokens, prompt tokens, elapsed time, output identity, and cache configuration |
| Evidence | `native-backend` |

**Experiment:** Run cold, warm-exact, and mutated-prefix requests through one APC-enabled native engine.


## 5. Inspect the experiment code

The code keeps one engine alive so the second request can reuse resident blocks. It introspects cache-related metrics and records `None` when the installed API does not expose a field.

Do not execute until the code matches the frozen table.


In [2]:
prefix=("vLLM stores key/value vectors in fixed-size cache blocks for scheduled token work. "*90)
llm=LLM(**base_engine_args(max_model_len=2048,enable_prefix_caching=True))
params=SamplingParams(temperature=0.0,max_tokens=8,seed=SEED)
def apc(prompt):
    tick=time.perf_counter(); item=llm.generate([prompt],params,use_tqdm=False)[0]; row=output_record(item)
    row["elapsed_s"]=time.perf_counter()-tick; row["cached_tokens"]=int(getattr(item,"num_cached_tokens",0) or 0)
    return row
cold=apc(prefix+" Question: define Prefill."); warm=apc(prefix+" Question: define Decode.")
mutated=apc("Changed. "+prefix+" Question: define Decode.")
metrics={"cold":cold,"warm":warm,"mutated":mutated,"prefix_characters":len(prefix),"enable_prefix_caching":True}
analysis=(f"Cold/warm/mutated requests reported {cold['cached_tokens']}/{warm['cached_tokens']}/"
          f"{mutated['cached_tokens']} cached tokens; warm/cold elapsed was {warm['elapsed_s']:.4f}/"
          f"{cold['elapsed_s']:.4f} s. The cached-token field is the hit evidence.")


INFO 08-13 00:19:51 [api_utils.py:273] non-default args: {'tokenizer': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct', 'dtype': 'bfloat16', 'seed': 20260824, 'max_model_len': 2048, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.45, 'max_num_seqs': 16, 'disable_log_stats': True, 'enforce_eager': True, 'model': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct'}


INFO 08-13 00:19:51 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 08-13 00:19:51 [model.py:1883] Using max model len 2048


INFO 08-13 00:19:51 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.


WARNING 08-13 00:19:51 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none


...

(EngineCore pid=651229) 2026-08-13 00:20:03,824 - INFO - autotuner.py:2397 - flashinfer.jit: [Autotuner]: Loaded 0 configs from <remote-home>/.cache/vllm/flashinfer_autotune_cache/flashinfer/0.6.16.post3/a10ba88b4090547465f2afcd30e11488df4381edf2bc37e4f4901ce2732cfd44/autotune_configs.json
(EngineCore pid=651229) 2026-08-13 00:20:03,825 - INFO - autotuner.py:829 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=651229) 2026-08-13 00:20:03,886 - INFO - autotuner.py:852 - flashinfer.jit: [Autotuner]: Autotuning process ends
(EngineCore pid=651229) 2026-08-13 00:20:03,894 - INFO - autotuner.py:2269 - flashinfer.jit: [Autotuner]: Saved 0 configs to <remote-home>/.cache/vllm/flashinfer_autotune_cache/flashinfer/0.6.16.post3/a10ba88b4090547465f2afcd30e11488df4381edf2bc37e4f4901ce2732cfd44/autotune_configs.json (0 new, 0 from previous config)


(EngineCore pid=651229) INFO 08-13 00:20:04 [jit_monitor.py:79] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.


(EngineCore pid=651229) INFO 08-13 00:20:04 [core.py:355] init engine (profile, create kv cache, warmup model) took 2.92 s


(EngineCore pid=651229) WARNING 08-13 00:20:05 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=651229) WARNING 08-13 00:20:05 [vllm.py:1247] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=651229) INFO 08-13 00:20:05 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
(EngineCore pid=651229) INFO 08-13 00:20:05 [vllm.py:1426] Cudagraph is disabled under eager mode
(EngineCore pid=651229) INFO 08-13 00:20:05 [compilation.py:329] Enabled custom fusions: norm_quant, act_quant


INFO 08-13 00:20:05 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Cold cached tokens | 0 |
| Warm cached tokens | 1,520 |
| Mutated cached tokens | 0 |
| Cold elapsed | 0.161883 |
| Warm elapsed | 0.079997 |
| Warm output tokens | 8 |


## 7. Explain the result

Cold/warm/mutated requests reported 0/1520/0 cached tokens; warm/cold elapsed was 0.0800/0.1619 s. The cached-token field is the hit evidence.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`native-backend`**. The named vLLM runtime executed on the recorded GPU/model/workload. The result does not transfer to another version, model, endpoint, or traffic distribution.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 12, "title": 'Automatic Prefix Caching', "environment": ENV,
    "evidence_label": 'native-backend', "metrics": metrics,
    "analysis": analysis, "conclusion": 'APC reuses exact, valid KV blocks; the retained native metadata distinguishes an observed hit from a timing guess.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 12,
  "title": "Automatic Prefix Caching",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260824
  },
  "evidence_label": "native-backend",
  "metrics": {
    "cold": {
      "request_id": "0",
      "prompt_tokens": 1537,
      "output_tokens": 8,
      "token_ids": [
        220,
        21806,
        25,
        348,
        4086,
        44,
        10533,
        1376
      ],
      "text_preview": "  Answer: vLLM stores key",
      "text_sha256": "07fd2a0f0c58b59a504ca6eaa79568d9182889eff21c2588cf06cf0dfffe1b6a",
      "finish_reason": "length",
      "stop_reason": null,
      "num_cached_tokens": 0,
      "elapsed_s": 0.16188318887725472,
      "cached_tokens": 0
    },
    "warm": {
      "request_id": "1",
      "prompt_tokens": 1536,
      "output_tokens": 8,


## 9. Make the bounded decision

> APC reuses exact, valid KV blocks; the retained native metadata distinguishes an observed hit from a timing guess.

**Acceptance/rollback:** Enable APC for a route only when production prefixes repeat, correctness is unchanged, and hit-rate/latency improve without harmful cache pressure.

**Failure analysis:** Very short prefixes, low repetition, eviction, multimodal hashes, LoRA identity, or non-deterministic prompt construction can eliminate reuse. Elapsed time alone does not prove a cache hit.


## 10. Extend the evidence

Instrument prefix-cache query/hit metrics under a real trace, then segment results by prefix family, block alignment, eviction age, and tenant boundary.

The full boundary and references are in [`README.md`](README.md).
